# Data11 frequency resolution analysis

Run all cells from the release environment. Only IM_6-14 are analyzed: IM_6/7/8 at 100 Hz, IM_9/10/11 at 100.01 Hz and IM_12/13/14 at 100.005 Hz. Each setting has three independent recordings. The primary window is [0,50) s and the trimmed comparison is [5,45) s. Extra duration in longer files is not an additional repeat. Actual windows, sample counts, timing and FFT grids are exported.

Native FFT bins, three/five-point interpolation, 8x padded bins/interpolation and raw-timestamp sine fitting are retained separately. The displayed global-peak method searches strictly f > 1 Hz through Nyquist; this is a search bound, not a high-pass filter. Original f > 0 global estimates and 99-101 Hz target-band estimates remain separate. Sine fitting refines the observed native peak locally. A stronger harmonic or other component can still be selected.

Independent estimates, means and sample SD are shown; unpaired mean differences and exploratory Welch 95% intervals are also saved. A 50 s record has a native grid near 0.02 Hz. Zero padding does not increase duration or establish simultaneous two-tone resolution at 0.005 Hz. Single-tone repeatability, distinguishability of settings and two-tone resolution are different claims.

Nominal values are generator settings, not calibrated frequency truth. Absolute/ppm estimates do not correct unknown clock bias. Relative plots subtract the three-record 100 Hz mean separately for each method/window as a display convention. Timestamp accuracy, analog bandwidth and batch drift remain unresolved.


In [ ]:
# data11-cell: setup
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display, Image, Markdown

for _candidate in (Path.cwd(), *Path.cwd().parents):
    if (_candidate / "release_manifest.json").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
from portable_runtime import project_root, data_root, ensure_environment, notebook_path
PROJECT = project_root()
ensure_environment()
NOTEBOOK_PATH = notebook_path("data11_resolution_analysis")
from data11_config import default_config
from data11_analysis import manifest, run_analysis, METHODS

pd.set_option("display.max_columns", 24)
pd.set_option("display.precision", 8)
print("Python:", sys.executable)

# Reload the shared paper figure style for this Run All.
import importlib as _paper_importlib
import paper_figure_style as _paper_style
_paper_importlib.reload(_paper_style)


## Configuration

These selections apply to Data11. Save the notebook after editing the settings.


In [ ]:
# data11-cell: configuration
CONFIG = default_config("resolution")
CONFIG["data_root"] = str(data_root("Data11"))
CONFIG["accel_g_per_count"] = 0.000488  # Reported accelerometer range: ±16 g
CONFIG["reference_conditions_confirmed"] = True
CONFIG["windows_s"] = {"primary_50s": [0.0, 50.0], "trimmed_40s": [5.0, 45.0]}
CONFIG["fft_pad_factor"] = 8
CONFIG["global_search_min_hz"] = 1.0  # Strictly f > cutoff, up to Nyquist; no high-pass filter
CONFIG["resolution_fit_example"] = "IM_12.csv"
# Square relative-frequency figures for both analysis windows.
CONFIG["resolution_relative_square"] = True
CONFIG["resolution_relative_figure_size_in"] = 10.0  # Base side in inches; reserve legend/method space
display(pd.DataFrame(CONFIG["files"].items(), columns=["file", "nominal_hz"]))


## Inputs and methods

Check input roles, paths and missing files. This table does not analyze or modify raw CSV files. The run records the full parameters and methods.


In [ ]:
# data11-cell: preview
INPUT_PLAN = pd.DataFrame(manifest(CONFIG))
INPUT_PLAN["exists"] = INPUT_PLAN.path.map(lambda p: Path(p).is_file())
display(INPUT_PLAN)
for key, text in METHODS.items():
    if key in {"input", "timing", "reference", CONFIG["analysis"]}:
        print(key + ": " + text + "\n")


## Run

Each execution creates a new dated run under the configured results directory. Existing runs and raw data are preserved. Failures retain partial outputs and an explicit status.


In [ ]:
# data11-cell: run
# Reload plotting code so rerunning this cell uses the current square layout.
import importlib
import data11_resolution_plots
importlib.reload(data11_resolution_plots)
shell = get_ipython()
history = shell.history_manager.input_hist_raw if shell is not None else []
live_cells = [s for s in history if s.lstrip().startswith("# data11-cell:")]
RESULTS = run_analysis(CONFIG, NOTEBOOK_PATH, executed_cells=live_cells)
OUTPUT_PATH = Path(RESULTS["output_path"])
print("Completed:", OUTPUT_PATH)


## Results

Sample SD describes variation between independent recordings. Interpret estimates alongside the quality audit and actual supported windows.


In [ ]:
# data11-cell: tables
print("summary")
display(RESULTS["summary"])
print("contrasts")
display(RESULTS["contrasts"])


In [ ]:
# data11-cell: figures
def show_figures(*keywords):
    for path in RESULTS["figures"]:
        if any(word in Path(path).stem for word in keywords):
            display(Image(filename=path))
show_figures("_comparison")

## Global-peak supplementary figures

These plots use the largest FFT peak strictly above `global_search_min_hz` (1 Hz) through Nyquist. Stronger harmonics are not replaced with a peak near 100 Hz. Original `global_*` and `target_*` results remain separate.

1. Relative frequency: subtract the three-record 100 Hz mean separately by method and window.
2. Uncorrected ppm: `1e6 * (estimated - nominal) / nominal`, without baseline or clock correction.
3. Spectra and seven estimators: normalize each displayed spectrum individually and retain all record estimates.
4. Three-point example: IM_12 shows native FFT points, the log-amplitude quadratic fit and padded points.

The default display is the 50 s primary window; the 40 s comparison is also saved. Error bars are sample SD (`ddof=1`), not confidence intervals. Normalization and baseline subtraction are display choices; this compares single tones in separate recordings.


In [ ]:
# data11-cell: resolution_detail_figures
RESOLUTION_DISPLAY_WINDOW = "primary_50s"  # Optional "trimmed_40s" display
for figure_type in ("relative_frequency", "absolute_ppm", "spectra_estimators", "quadratic_example"):
    show_figures(f"resolution_{RESOLUTION_DISPLAY_WINDOW}_{figure_type}")
print("New figures and CSVs:", RESULTS["output_path"])


## Quality and output files

`dt_cv` describes sampling-interval variation. A recording-level gap may lie outside an analyzed window; inspect window-level `gap_ok` as well. All per-recording diagnostics are saved in the output directory.


In [ ]:
# data11-cell: diagnostics
display(RESULTS["audit"])
display(pd.DataFrame(RESULTS["input_issues"]))
print("Exact output path:", OUTPUT_PATH)
print("PNG / CSV / parameters.json / inputs.json / provenance.json / methods.txt / run_status.json / scripts/")
print("Limitations: uncalibrated timebase; nominal accelerometer conversion; independent batches; inspect excluded windows and records.")


## Method references

Data11 uses its own parameters and functions. Related Data10 notebooks retain their separate metric definitions. Implementation references: [SciPy Hann window](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.windows.hann.html) and [SciPy spectrogram](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.spectrogram.html).
